# 05c - Gold+weak checkpoint training (seed=42 only)

Companion notebook to `05b_gold_weak_training.ipynb`, built to answer a
different question: **is our 58-gold local CV number trustworthy at all**,
by getting a first real leaderboard score. `05b` never saved any model
weights (`del model` after every fold) so there is nothing to submit yet.

This notebook reuses Cells 1-6 of `05b` **verbatim, unchanged** (already
validated against real Kaggle output in that notebook) and only trains
`gold_weak`/`seed=42` across the 5 folds -- the arm+seed combination
`05b` already measured at pooled OOF macro-AUC 0.5659 (val) /
train_auc_mean 0.7871 -- adding one thing: it saves each fold's trained
weights to `/kaggle/working/checkpoints/fold{k}.pth` instead of deleting
the model. Those 5 checkpoints get packaged as a new private Kaggle
Dataset (manual step, same as `05a`'s dataset publish) for
`06_submission_inference.ipynb` to mount and ensemble.

Not repeating `gold_only` or seeds 43/44 here -- `05b` already answered
those, no need to spend Kaggle GPU-hours re-deriving a number we have.

## Cell 1 - Imports, mount, constants

Identical to `05b` Cell 1, except `SEEDS`/`ARMS` are trimmed to the single (arm, seed) this notebook trains.

In [ ]:
# Self-contained (no `from src import ...`) -- Kaggle doesn't mount this repo.
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

RAW_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
TRIPLETS_DIR = Path("/kaggle/input/datasets/alherma7/triplets-knee")
assert RAW_DIR.exists(), f"Competition data not found at {RAW_DIR}"
assert TRIPLETS_DIR.exists(), f"Triplets dataset not found at {TRIPLETS_DIR}"

npy_files = list(TRIPLETS_DIR.glob("*.npy"))
print(f"Triplet files found: {len(npy_files)}")
assert len(npy_files) == 4407, f"Expected 4407 triplet files, found {len(npy_files)}"

RANDOM_STATE = 42
CV_FOLDS = 5
ARM = "gold_weak"
SEED = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

FINDINGS = [
    "acl_injury", "mcl_injury", "medial_meniscus_tear", "lateral_meniscus_tear",
    "oa_medial_compartment", "oa_lateral_compartment", "oa_patellofemoral_compartment",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("timm:", timm.__version__)

## Cell 2 - Label/fold table

Identical to `05b` Cell 2 (ported `src/labelers.py`/`src/data.py::load_training_labels`, graduated 2026-08-18).

In [ ]:
# Ported from src/labelers.py (graduated 2026-08-18) -- same logic, self-contained.
import hashlib
import re
import unicodedata


def _normalize(text):
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = unicodedata.normalize("NFKD", t)
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    t = re.sub(r"[_\-/\\]+", " ", t)
    t = re.sub(r"[ \t]+", " ", t)
    return t


_BULLET_PREFIX_RE = re.compile(r"^[>*\u2022\-]+\s*")


def _unwrap(text):
    if not isinstance(text, str):
        return ""
    out = []
    for line in text.split("\n"):
        s = line.strip()
        s_check = _BULLET_PREFIX_RE.sub("", s)
        if (out and out[-1] and not re.search(r"[.;:!?>*\u2022]$", out[-1])
                and len(out[-1].split()) >= 4 and s_check and not s_check[:1].isupper()):
            out[-1] = out[-1] + " " + s
        else:
            out.append(s)
    return "\n".join(out)


_SENT_SPLIT_RE = re.compile(r"(?<=[.;!?])\s*|\n+")


def _clauses(text):
    norm = _normalize(_unwrap(text))
    return [c.strip() for c in _SENT_SPLIT_RE.split(norm) if c and c.strip()]


_SUB_CLAUSE_SPLIT_RE = re.compile(r"\s+but\s+|\s+pero\s+|\s+aunque\s+|\s+although\s+")


def _sub_clauses(clause):
    return [s.strip() for s in _SUB_CLAUSE_SPLIT_RE.split(clause) if s and s.strip()]


_NEGATION_SCOPE_CUES = [
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\babsent\b",
    r"no evidence of", r"no sign", r"negative for",
    r"\bsin\b", r"\bausen", r"\bninguna\b", r"\bnegativ",
    r"no se (?:observa|evidencia|aprecia)",
]
_NEGATION_PREDICATE_CUES = [
    r"\bnormal\b", r"\bintact\b", r"\bunremarkable\b", r"within normal limit",
    r"dentro de (los )?l[i0]mites normales",
]
_NEGATION_SCOPE_RE = re.compile("|".join(_NEGATION_SCOPE_CUES))
_NEGATION_PREDICATE_RE = re.compile("|".join(_NEGATION_PREDICATE_CUES))


def _negation_applies(sub_clause, anatomy_re, pathology_re):
    for m in _NEGATION_SCOPE_RE.finditer(sub_clause):
        arg = sub_clause[m.end():]
        if anatomy_re.search(arg) or pathology_re.search(arg):
            return True
    for m in _NEGATION_PREDICATE_RE.finditer(sub_clause):
        arg = sub_clause[:m.start()]
        if anatomy_re.search(arg) or pathology_re.search(arg):
            return True
    return False


_OA_PATHOLOGY_CUES = [
    r"osteoarthrit", r"osteoarthros", r"osteoartr", r"chondrosis",
    r"(cartilage|chondral) (loss|thinning|defect|fissur)",
    r"joint space narrowing", r"osteophyte", r"osteofito", r"spurring",
    r"pinzamiento", r"degenerat", r"adelgazamiento del cartilago",
    r"chondromalacia", r"condromalacia", r"subchondral cystic",
]

FINDING_LEXICON = {
    "acl_injury": dict(
        anatomy=[r"\bacl\b", r"anterior cruciate ligament", r"ligamento cruzado anterior", r"\blca\b"],
        pathology=[r"\btear", r"\btorn\b", r"ruptur", r"sprain", r"rotur", r"desgarr", r"esguinc", r"discontinuit"],
    ),
    "mcl_injury": dict(
        anatomy=[r"\bmcl\b", r"medial collateral ligament", r"ligamento colateral medial", r"ligamento lateral interno"],
        pathology=[r"\btear", r"\btorn\b", r"ruptur", r"sprain", r"rotur", r"desgarr", r"esguinc"],
    ),
    "medial_meniscus_tear": dict(
        anatomy=[r"medial meniscus", r"menisco medial"],
        pathology=[r"\btear", r"\btorn\b", r"rotur", r"desgarr", r"extrusion", r"extrusi[o0]n", r"macerat"],
    ),
    "lateral_meniscus_tear": dict(
        anatomy=[r"lateral meniscus", r"menisco lateral"],
        pathology=[r"\btear", r"\btorn\b", r"rotur", r"desgarr", r"extrusion", r"extrusi[o0]n", r"macerat"],
    ),
    "oa_medial_compartment": dict(
        anatomy=[r"medial compartment", r"medial femorotibial", r"medial femoral condyle",
                 r"medial tibial plateau", r"compartimento medial"],
        pathology=_OA_PATHOLOGY_CUES,
    ),
    "oa_lateral_compartment": dict(
        anatomy=[r"lateral compartment", r"lateral femorotibial", r"lateral femoral condyle",
                 r"lateral tibial plateau", r"compartimento lateral"],
        pathology=_OA_PATHOLOGY_CUES,
    ),
    "oa_patellofemoral_compartment": dict(
        anatomy=[r"patellofemoral", r"femoropatelar", r"femororrotulian", r"patelofemoral"],
        pathology=_OA_PATHOLOGY_CUES,
    ),
    "effusion": dict(
        anatomy=[r"\beffusion\b", r"derrame articular", r"\bderrame\b", r"efusi[o0]n"],
        pathology=[r"\beffusion\b", r"derrame", r"efusi[o0]n", r"fluid collection", r"joint fluid"],
    ),
    "synovitis": dict(
        anatomy=[r"synovit", r"sinovit", r"synovial (thickening|hypertrophy|proliferation)", r"engrosamiento sinovial"],
        pathology=[r"synovit", r"sinovit", r"synovial (thickening|hypertrophy|proliferation)", r"engrosamiento sinovial"],
    ),
    "bakers_cyst": dict(
        anatomy=[r"baker'?s? cyst", r"popliteal cyst", r"quiste de baker", r"quiste poplite"],
        pathology=[r"baker'?s? cyst", r"popliteal cyst", r"quiste de baker", r"quiste poplite"],
    ),
    "bone_contusion": dict(
        anatomy=[r"bone (marrow )?contusion", r"bone (marrow )?edema", r"contusi[o0]n [o0]sea", r"edema [o0]seo", r"edema medular"],
        pathology=[r"bone (marrow )?contusion", r"bone (marrow )?edema", r"contusi[o0]n [o0]sea", r"edema [o0]seo", r"edema medular"],
    ),
    "fracture": dict(
        anatomy=[r"\bfractur"],
        pathology=[r"\bfractur", r"cortical (break|disruption)", r"trabecular fracture"],
    ),
}

_COMPILED_LEXICON = {
    finding: (re.compile("|".join(cues["anatomy"])), re.compile("|".join(cues["pathology"])))
    for finding, cues in FINDING_LEXICON.items()
}


def label_report(report_text, finding):
    anatomy_re, pathology_re = _COMPILED_LEXICON[finding]
    votes = []
    for clause in _clauses(report_text):
        for sub in _sub_clauses(clause):
            if not anatomy_re.search(sub):
                continue
            if _negation_applies(sub, anatomy_re, pathology_re):
                votes.append(0.0)
            elif pathology_re.search(sub):
                votes.append(1.0)
    if not votes:
        return 0.5
    return 1.0 if max(votes) == 1.0 else 0.0


def label_reports(reports, findings):
    if "StudyInstanceUID" in reports.columns:
        reports = reports.set_index("StudyInstanceUID")
    return pd.DataFrame({
        finding: reports["Report"].apply(lambda text: label_report(text, finding))
        for finding in findings
    }, index=reports.index)


def report_group_key(report_text):
    if not isinstance(report_text, str):
        normalized = ""
    else:
        t = unicodedata.normalize("NFKD", report_text.lower())
        t = "".join(ch for ch in t if not unicodedata.combining(ch))
        normalized = re.sub(r"\s+", " ", t).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


# --- load_training_labels (ported from src/data.py) ---
train = pd.read_csv(RAW_DIR / "train.csv")
reports = train[["StudyInstanceUID", "Report"]].set_index("StudyInstanceUID")

gold_mask = train[LABEL_COLS].notna().all(axis=1)
gold = train.loc[gold_mask, ["StudyInstanceUID"] + LABEL_COLS].set_index("StudyInstanceUID")
gold.columns = FINDINGS
assert len(gold) == 58, f"Expected 58 gold studies, found {len(gold)}"

is_gold = reports.index.isin(gold.index)
weak_reports = reports.loc[~is_gold].reset_index()
weak_labels = label_reports(weak_reports, FINDINGS)

combined = pd.concat([gold[FINDINGS], weak_labels[FINDINGS]])
combined = combined.loc[reports.index]
combined["is_gold"] = is_gold

group_keys = reports["Report"].apply(report_group_key)
gkf = GroupKFold(n_splits=CV_FOLDS)
fold = pd.Series(-1, index=reports.index, dtype=int)
for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=group_keys.to_numpy())):
    fold.iloc[val_idx] = fold_idx
combined["fold"] = fold

label_table = combined
print(f"label_table: {len(label_table)} rows, {int(label_table['is_gold'].sum())} gold")
print("Gold per fold:")
print(label_table.loc[label_table["is_gold"], "fold"].value_counts().sort_index())

## Cell 3 - Dataset / DataLoader

Identical to `05b` Cell 3.

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, study_ids, label_table, triplets_dir):
        self.study_ids = list(study_ids)
        self.label_table = label_table
        self.triplets_dir = triplets_dir

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        study_id = self.study_ids[idx]
        triplet = np.load(self.triplets_dir / f"{study_id}.npy")
        target = self.label_table.loc[study_id, FINDINGS].to_numpy(dtype=np.float32)
        is_gold = bool(self.label_table.loc[study_id, "is_gold"])
        return torch.from_numpy(triplet), torch.from_numpy(target), is_gold, study_id


available_ids = {p.stem for p in npy_files}
missing = set(label_table.index) - available_ids
print(f"Studies missing a triplet file: {len(missing)}")
label_table = label_table.loc[label_table.index.isin(available_ids)]
print(f"label_table after intersection: {len(label_table)} rows")

## Cell 4 - Model (EfficientNet-B0 + differential-LR head)

Identical to `05b` Cell 4.

In [ ]:
class BaselineFindingModel(nn.Module):
    def __init__(self, backbone_name, n_findings, dropout=0.5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, n_findings),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def differential_lr_param_groups(model, backbone_lr, head_lr):
    return [
        {"params": list(model.backbone.parameters()), "lr": backbone_lr},
        {"params": list(model.head.parameters()), "lr": head_lr},
    ]

## Cell 5 - Loss with corrected `pos_weight`

Identical to `05b` Cell 5.

In [ ]:
def compute_pos_weight(targets_df):
    """n_negative / n_positive per finding, over hard-labeled rows only."""
    weights = []
    for col in FINDINGS:
        hard = targets_df[col][targets_df[col] != 0.5]
        n_pos = max((hard == 1.0).sum(), 1)
        n_neg = (hard == 0.0).sum()
        weights.append(n_neg / n_pos)
    return torch.tensor(weights, dtype=torch.float32)

## Cell 6 - Single (seed, fold) training function, with checkpoint saving

Same training logic as `05b` Cell 6 (`batch_size=48` for `gold_weak`, 8
fixed epochs, no val-based checkpoint selection), **plus** one change:
after training, the model's `state_dict()` (moved to CPU) is saved to
`checkpoint_path` before it's freed -- `05b` never did this, it always
deleted the model.

In [ ]:
def macro_roc_auc(y_true_df, y_pred_df):
    aucs = [roc_auc_score(y_true_df[col], y_pred_df[col]) for col in y_true_df.columns]
    return float(np.mean(aucs))


N_EPOCHS = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def run_fold_and_save(seed, fold_idx, label_table, triplets_dir, checkpoint_path, n_epochs=N_EPOCHS):
    torch.manual_seed(seed)

    train_mask = label_table["fold"] != fold_idx
    train_ids = label_table.index[train_mask]

    val_mask = (label_table["fold"] == fold_idx) & label_table["is_gold"]
    val_ids = label_table.index[val_mask]

    batch_size = 48
    eval_batch_size = 64

    train_ds = TripletDataset(train_ids, label_table, triplets_dir)
    val_ds = TripletDataset(val_ids, label_table, triplets_dir)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=eval_batch_size, shuffle=False, num_workers=2, pin_memory=True)

    train_pos_weight = compute_pos_weight(label_table.loc[train_ids, FINDINGS]).to(DEVICE)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=train_pos_weight)

    model = BaselineFindingModel("efficientnet_b0", n_findings=len(FINDINGS), pretrained=True).to(DEVICE)
    optimizer = torch.optim.Adam(
        differential_lr_param_groups(model, backbone_lr=1e-5, head_lr=1e-3),
        weight_decay=1e-2,
    )

    for epoch in range(n_epochs):
        model.train()
        epoch_loss, n_seen = 0.0, 0
        for xb, yb, _, _ in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
            n_seen += len(xb)
        print(f"    [seed{seed}/fold{fold_idx}] epoch {epoch+1}/{n_epochs}: loss={epoch_loss/n_seen:.4f}")

    model.eval()

    def predict(loader):
        preds, ids = [], []
        with torch.no_grad():
            for xb, _, _, batch_ids in loader:
                xb = xb.to(DEVICE)
                probs = torch.sigmoid(model(xb)).cpu().numpy()
                preds.append(probs)
                ids.extend(batch_ids)
        return pd.DataFrame(np.concatenate(preds), index=ids, columns=FINDINGS)

    val_preds = predict(val_loader)

    # New vs. 05b: persist the trained weights instead of discarding them.
    torch.save({k: v.cpu() for k, v in model.state_dict().items()}, checkpoint_path)
    print(f"    saved checkpoint -> {checkpoint_path}")

    del model
    torch.cuda.empty_cache()

    return val_preds


# Smoke test (1 epoch, fold 0) -- cheap sanity check before the full 5-fold loop.
_t0 = time.time()
_smoke_path = CHECKPOINT_DIR / "smoke_test.pth"
_val_p = run_fold_and_save(
    seed=SEED, fold_idx=0, label_table=label_table, triplets_dir=TRIPLETS_DIR,
    checkpoint_path=_smoke_path, n_epochs=1,
)
print(f"Smoke test done in {time.time() - _t0:.1f}s: val_ids={len(_val_p)}")
assert _smoke_path.exists() and _smoke_path.stat().st_size > 0, "Checkpoint file was not written"
_smoke_path.unlink()  # discard -- this was only a 1-epoch smoke test, not a real fold

## Cell 7 - Train and save all 5 folds

Trains `gold_weak`/`seed=42` for each of the 5 folds, saving
`checkpoints/fold{k}.pth`, and pools the 5 folds' val predictions into
one 58-row table to compute the same pooled OOF macro-AUC `05b` already
measured for this exact (arm, seed) -- **expected: val_macro_auc ~=
0.5659** (05b's reported number for `gold_weak`/seed=42). This is the
sanity check that retraining-with-checkpointing reproduces the
already-validated result before trusting these checkpoints for a real
submission.

In [ ]:
CV_FOLDS_RANGE = range(CV_FOLDS)

fold_val_preds = []
t_start = time.time()
for fold_idx in CV_FOLDS_RANGE:
    checkpoint_path = CHECKPOINT_DIR / f"fold{fold_idx}.pth"
    val_preds = run_fold_and_save(
        seed=SEED, fold_idx=fold_idx, label_table=label_table, triplets_dir=TRIPLETS_DIR,
        checkpoint_path=checkpoint_path, n_epochs=N_EPOCHS,
    )
    fold_val_preds.append(val_preds)
    print(f"  fold={fold_idx}: val_n={len(val_preds)}")

val_pooled = pd.concat(fold_val_preds)
assert len(val_pooled) == 58, f"Expected 58 pooled gold val rows, got {len(val_pooled)}"
assert val_pooled.index.is_unique, "Duplicate study in pooled val predictions"

val_true = label_table.loc[val_pooled.index, FINDINGS]
val_macro = macro_roc_auc(val_true, val_pooled)
print(f"\nPooled OOF macro-AUC (seed={SEED}, 5 folds): {val_macro:.4f}")
print("05b's reference number for this exact (arm=gold_weak, seed=42): 0.5659")
print(f"Delta: {val_macro - 0.5659:+.4f}")
print(f"Total wall time: {(time.time() - t_start) / 60:.1f} min")

_checkpoint_files = sorted(CHECKPOINT_DIR.glob("fold*.pth"))
print(f"\nCheckpoints saved: {len(_checkpoint_files)}")
for f in _checkpoint_files:
    print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")
assert len(_checkpoint_files) == CV_FOLDS, f"Expected {CV_FOLDS} checkpoint files, found {len(_checkpoint_files)}"

## Cell 8 - Package checkpoints for the inference notebook

Same manual publish pattern as `05a` Cell 12 (in-kernel Notebook Output
Files didn't surface reliably there either): zip the checkpoints dir,
plus a small metadata file recording the finding order and
hyperparameters the inference notebook needs to reconstruct the model
architecture, then publish as a new private Kaggle Dataset. This upload
step is the user's job (Kaggle-side action), not run automatically.

In [ ]:
import json
import shutil

metadata = {
    "arm": ARM,
    "seed": SEED,
    "cv_folds": CV_FOLDS,
    "findings": FINDINGS,
    "official_label_columns": OFFICIAL_LABEL_COLUMNS,
    "backbone": "efficientnet_b0",
    "n_epochs": N_EPOCHS,
    "reference_pooled_oof_macro_auc": val_macro,
}
with open(CHECKPOINT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

archive_path = shutil.make_archive("/kaggle/working/gold_weak_seed42_checkpoints", "zip", CHECKPOINT_DIR)
print(f"Archive written: {archive_path} ({Path(archive_path).stat().st_size / 1e6:.1f} MB)")
print()
print("Next (manual, Kaggle-side): download this zip (Quick Save -> kaggle kernels output,")
print("or the Output tab), then publish it as a new private Kaggle Dataset -- e.g.")
print("'gold-weak-seed42-checkpoints' -- for 06_submission_inference.ipynb to mount.")

## Closing task (after this notebook finishes)

Report the pooled OOF macro-AUC printed in Cell 7 (should land close to
0.5659, `05b`'s number for this exact arm/seed) and confirm the 5
checkpoint files + `metadata.json` look right before publishing the
Kaggle Dataset. Once published, note the dataset's actual mount slug
(discovered empirically, same as `triplets_knee` -> `triplets-knee` in
`05b` Cell 1) -- `06_submission_inference.ipynb`'s Cell 1 needs it.